# Assignment 4

This is an basecode for assignment 4 of Artificial Intelligence class (CSCE-4613), Spring 2025


In [1]:
import torch
import torch.nn as nn
import torchvision

## Binary Network

## Define a binary network class


In [2]:
class BinaryNetwork(nn.Module):
  def __init__(self):
    # YOUR CODE HERE
    super(BinaryNetwork, self).__init__()
    self.fc1 = nn.Linear(2, 4)
    self.relu = nn.ReLU()
    self.fc2 = nn.Linear(4, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, x):
    # x has the size of (batch size x 2)
    # YOUR CODE HERE
    x = self.fc1(x)
    x = self.relu(x)
    x = self.fc2(x)
    x = self.sigmoid(x)
    return x

### Define data generator

In [3]:
def generate_data(operator = "AND"):
  assert operator in ["AND", "OR", "XOR", "NOR"], "%s operator is not valid" % operator
  data = []
  label = []
  for i in range(2):
    for j in range(2):
      data.append([i, j])
      if operator == "AND":
        label.append(i & j)
      elif operator == "OR":
        label.append(i | j)
      elif operator == "XOR":
        label.append(i ^ j)
      else:
        label.append(not (i | j))
  data = torch.as_tensor(data, dtype = torch.float32)
  label = torch.as_tensor(label, dtype = torch.float32)
  return data, label

### Define the training framework

In [6]:
model = BinaryNetwork()
model.train()
print(model)
operator = "XOR"
inputs, labels = generate_data(operator = operator)

n_iters = 1000
learning_rate = 0.1
loss_fn = nn.BCELoss()
losses = []

optim = torch.optim.SGD(params = model.parameters(), lr = learning_rate, momentum=0.9)


for i in range(1, n_iters + 1):

  # WRITE YOUR CODE TO COMPUTE OUTPUTS, LOSS, ACCURACY, AND OPTIMIZE MODEL
  # outptus = ???
  # loss = ???
  # accuracy = ???
  outputs = model(inputs).squeeze() 
  loss = loss_fn(outputs, labels)
  preds = (outputs >= 0.5).float()
  accuracy = (preds == labels).float().mean().item() * 100

  
  # optimize the model
  optim.zero_grad()
  loss.backward()
  optim.step()
  losses.append(loss.item())

  if i % 5 == 0:
    print("[%d/%d]. Loss: %0.4f. Accuracy: %0.2f" % (i, n_iters, loss.item(), accuracy))

model.eval()
# WRITE YOUR CODE TO CALCULATE THE FINAL ACCURACY
# accuracy = ???
with torch.no_grad():
    outputs = model(inputs).squeeze() 
    preds = (outputs >= 0.5).float()
    accuracy = (preds == labels).float().mean().item() * 100
print("Final Accuracy: %0.2f" % (accuracy))

torch.save(model.state_dict(), "%s_Network.pth" % operator)
  # model.load_state_dict(torch.load("%s_Network.pth" % operator)) # Load model in the next time you use

BinaryNetwork(
  (fc1): Linear(in_features=2, out_features=4, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=4, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)
[5/1000]. Loss: 0.6856. Accuracy: 50.00
[10/1000]. Loss: 0.6713. Accuracy: 50.00
[15/1000]. Loss: 0.6570. Accuracy: 75.00
[20/1000]. Loss: 0.6354. Accuracy: 75.00
[25/1000]. Loss: 0.6151. Accuracy: 75.00
[30/1000]. Loss: 0.5854. Accuracy: 75.00
[35/1000]. Loss: 0.5565. Accuracy: 75.00
[40/1000]. Loss: 0.5277. Accuracy: 75.00
[45/1000]. Loss: 0.5079. Accuracy: 75.00
[50/1000]. Loss: 0.4961. Accuracy: 75.00
[55/1000]. Loss: 0.4894. Accuracy: 75.00
[60/1000]. Loss: 0.4845. Accuracy: 75.00
[65/1000]. Loss: 0.4790. Accuracy: 75.00
[70/1000]. Loss: 0.4734. Accuracy: 75.00
[75/1000]. Loss: 0.4657. Accuracy: 75.00
[80/1000]. Loss: 0.4537. Accuracy: 75.00
[85/1000]. Loss: 0.4359. Accuracy: 75.00
[90/1000]. Loss: 0.4106. Accuracy: 75.00
[95/1000]. Loss: 0.3778. Accuracy: 100.00
[100/1000]. Loss: 0.3347. Accuracy: 100.00
[105/

## Digit Classification

### Define Digit Classification Network

In [12]:
class DigitNetwork(nn.Module):
  def __init__(self):
    super(DigitNetwork, self).__init__()
    # YOUR CODE HERE
    self.fc1 = nn.Linear(28 * 28, 128)
    self.relu = nn.ReLU()
    self.fc2 = nn.Linear(128, 10)

  def forward(self, x):
    # x has the size of (batch size x 1 x height x height)
    # YOUR CODE HERE
    x = x.view(x.size(0), -1) # flatten the input
    x = self.fc1(x)
    x = self.relu(x)
    x = self.fc2(x)
    return x

### Define Data Generator

In [13]:
def create_data_generator(batch_size = 32, root = "data"):
  train_dataset = torchvision.datasets.MNIST(root = root,
                                             train = True,
                                             transform = torchvision.transforms.ToTensor(),
                                             download = True)
  test_dataset = torchvision.datasets.MNIST(root = root,
                                             train = False,
                                             transform = torchvision.transforms.ToTensor(),
                                             download = True)
  train_loader = torch.utils.data.DataLoader(train_dataset,
                                             batch_size = batch_size,
                                             shuffle = True)
  test_loader = torch.utils.data.DataLoader(test_dataset,
                                             batch_size = batch_size,
                                             shuffle = False)
  return train_loader, test_loader

### Define the training framework

In [17]:
cuda = torch.cuda.is_available()
batch_size = 32
train_loader, test_loader = create_data_generator(batch_size)
model = DigitNetwork()
print(model)
if cuda:
  model.cuda()
n_epochs = 1
learning_rate = 0.1
optim = torch.optim.SGD(params = model.parameters(), lr = learning_rate, momentum=0.9)
loss_fn = nn.CrossEntropyLoss()

model.train()
for epoch in range(1, n_epochs + 1):
  for idx, (images, labels) in enumerate(train_loader):
    # WRITE YOUR CODE TO COMPUTE OUTPUTS, LOSS, ACCURACY, AND OPTIMIZE MODEL
    # outptus = ???
    # loss = ???
    # accuracy = ???
    outputs = model(images)
    loss = loss_fn(outputs, labels)
    preds = torch.argmax(outputs, dim = 1)
    accuracy = (preds == labels).float().mean().item() * 100
    # optimize the model
    optim.zero_grad()
    loss.backward()
    optim.step()

    if idx % 100 == 0:
      print("Epoch [%d/%d]. Iter [%d/%d]. Loss: %0.2f. Accuracy: %0.2f" % (epoch, n_epochs, idx + 1, len(train_loader), loss, accuracy))

torch.save(model.state_dict(), "MNIST_Network.pth")

DigitNetwork(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)
Epoch [1/1]. Iter [1/1875]. Loss: 2.28. Accuracy: 21.88
Epoch [1/1]. Iter [101/1875]. Loss: 0.15. Accuracy: 96.88
Epoch [1/1]. Iter [201/1875]. Loss: 0.34. Accuracy: 84.38
Epoch [1/1]. Iter [301/1875]. Loss: 0.37. Accuracy: 84.38
Epoch [1/1]. Iter [401/1875]. Loss: 0.51. Accuracy: 90.62
Epoch [1/1]. Iter [501/1875]. Loss: 0.26. Accuracy: 93.75
Epoch [1/1]. Iter [601/1875]. Loss: 0.36. Accuracy: 90.62
Epoch [1/1]. Iter [701/1875]. Loss: 0.20. Accuracy: 96.88
Epoch [1/1]. Iter [801/1875]. Loss: 0.22. Accuracy: 90.62
Epoch [1/1]. Iter [901/1875]. Loss: 0.38. Accuracy: 93.75
Epoch [1/1]. Iter [1001/1875]. Loss: 0.08. Accuracy: 96.88
Epoch [1/1]. Iter [1101/1875]. Loss: 0.11. Accuracy: 93.75
Epoch [1/1]. Iter [1201/1875]. Loss: 0.08. Accuracy: 93.75
Epoch [1/1]. Iter [1301/1875]. Loss: 0.15. Accuracy: 93.75
Epoch [1/1]. Iter [1401/1875].

### Define the evaluation framework

In [20]:
cuda = torch.cuda.is_available()
batch_size = 1
train_loader, test_loader = create_data_generator(batch_size)
model = DigitNetwork()
if cuda:
  model.cuda()
model.eval()
model.load_state_dict(torch.load("MNIST_Network.pth"))

total_accuracy = 0.0
accuracy = 0.0
with torch.no_grad():
  # WRITE YOUR CODE TO COMPUTE ACCURACY
  # accuracy = ???
  for idx, (images, labels) in enumerate(test_loader):
    if cuda:
      images = images.cuda()
      labels = labels.cuda()
    outputs = model(images)
    preds = torch.argmax(outputs, dim = 1)
    accuracy = (preds == labels).float().mean().item() * 100

    total_accuracy += accuracy

  if idx % 2000 == 0:
    print("Iter [%d/%d]. Accuracy: %0.2f" % (idx + 1, len(test_loader), accuracy))

print("Final Accuracy: %0.2f" % (total_accuracy / len(test_loader)))

Final Accuracy: 94.36


## Backpropagation

### ReLU Example

In [ ]:
# https://pytorch.org/tutorials/beginner/pytorch_with_examples.html#pytorch-defining-new-autograd-functions
class MyReLU(torch.autograd.Function):
    """
    We can implement our own custom autograd Functions by subclassing
    torch.autograd.Function and implementing the forward and backward passes
    which operate on Tensors.
    """

    @staticmethod
    def forward(ctx, input):
        """
        In the forward pass we receive a Tensor containing the input and return
        a Tensor containing the output. ctx is a context object that can be used
        to stash information for backward computation. You can cache arbitrary
        objects for use in the backward pass using the ctx.save_for_backward method.
        """
        ctx.save_for_backward(input)
        return input.clamp(min=0)

    @staticmethod
    def backward(ctx, grad_output):
        """
        In the backward pass we receive a Tensor containing the gradient of the loss
        with respect to the output, and we need to compute the gradient of the loss
        with respect to the input.
        """
        input, = ctx.saved_tensors
        grad_input = grad_output.clone()
        grad_input[input < 0] = 0
        return grad_input

#### Sigmoid Function


In [21]:
class MySigmoid(torch.autograd.Function):
    """
    We can implement our own custom autograd Functions by subclassing
    torch.autograd.Function and implementing the forward and backward passes
    which operate on Tensors.
    """

    @staticmethod
    def forward(ctx, input):
        # input is a N x C tensor, N is the batch size, C is the dimension of input
        ctx.save_for_backward(input)
        # YOUR CODE HERE
        # return output of sigmoid function
        output = 1 / (1 + torch.exp(-input))
        return output

    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors
        # YOUR CODE HERE
        # return grad_input
        sigmoid = 1 / (1 + torch.exp(-input))
        grad_input = grad_output * sigmoid * (1 - sigmoid)
        return grad_input

#### Fully Connected Layer

In [22]:
class MyLinearFunction(torch.autograd.Function):
    """
    We can implement our own custom autograd Functions by subclassing
    torch.autograd.Function and implementing the forward and backward passes
    which operate on Tensors.
    """

    @staticmethod
    def forward(ctx, input, weights, bias):
        # input is a N x C tensor, N is the batch size, C is the dimension of input
        # weights is a C x D tensor, C and D are the dimension out input and ouput
        # bias is D tensor
        ctx.save_for_backward(input, weights, bias)
        # YOUR CODE HERE
        # return output of linear function
        return torch.matmul(input, weights) + bias


    @staticmethod
    def backward(ctx, grad_output):
        input, weights, bias = ctx.saved_tensors
        # YOUR CODE HERE
        grad_input = torch.matmul(grad_output, weights.t())
        grad_weights = torch.matmul(input.t(), grad_output)
        grad_bias = torch.sum(grad_output, dim=0)
        # return grad_input, grad_weights, grad_bias
        return grad_input, grad_weights, grad_bias

class MyLinearLayer(nn.Module):
  # You don't modify this layer
  def __init__(self, in_features = 2, out_features = 4):
    super(MyLinearLayer, self).__init__()
    self.weights = nn.Parameter(torch.randn(in_features, out_features))
    self.bias = nn.Parameter(torch.zeros(out_features))
    self.linear_fn = MyLinearFunction.apply

  def forward(self, input):
    return self.linear_fn(input, self.weights, self.bias)


#### Testing Your Implementation

In [24]:
class MyLinearNetwork(nn.Module):
  def __init__(self):
    super(MyLinearNetwork, self).__init__()
    self.linear_1 = MyLinearLayer(28 * 28, 128)
    self.sigmoid_fn = MySigmoid.apply
    self.linear_2 = MyLinearLayer(128, 10)
    self.softmax_fn = nn.Softmax(dim=1)

  def forward(self, x):
    size = x.size()
    x = x.reshape(size[0], -1) # Flatten images
    x = self.linear_1(x)
    x = self.sigmoid_fn(x)
    x = self.linear_2(x)
    if self.training == False:
      x = self.softmax_fn(x)
    return x

In [26]:
cuda = torch.cuda.is_available()
batch_size = 32
train_loader, test_loader = create_data_generator(batch_size)
model = MyLinearNetwork()
print(model)
if cuda:
  model.cuda()
n_epochs = 3
learning_rate = 0.1
optim = torch.optim.SGD(params = model.parameters(), lr = learning_rate, momentum=0.9)
loss_fn = nn.CrossEntropyLoss()

model.train()
for epoch in range(1, n_epochs + 1):
  for idx, (images, labels) in enumerate(train_loader):
    # WRITE YOUR CODE TO COMPUTE OUTPUTS, LOSS, ACCURACY, AND OPTIMIZE MODEL
    # outptus = ???
    # loss = ???
    # accuracy = ???
    if cuda:
      images = images.cuda()
      labels = labels.cuda()

    outputs = model(images)
    loss = loss_fn(outputs, labels)
    preds = torch.argmax(outputs, dim = 1)
    accuracy = (preds == labels).float().mean().item() * 100
    # optimize the model
    optim.zero_grad()
    loss.backward()
    optim.step()  

    if idx % 100 == 0:
      print("Epoch [%d/%d]. Iter [%d/%d]. Loss: %0.2f. Accuracy: %0.2f" % (epoch, n_epochs, idx + 1, len(train_loader), loss, accuracy))

total_accuracy = 0.0
model.eval()
with torch.no_grad():
  for idx, (images, labels) in enumerate(test_loader):
    # WRITE YOUR CODE TO COMPUTE ACCURACY
    # accuracy = ???
    if cuda:
      images = images.cuda()
      labels = labels.cuda()
    outputs = model(images)
    preds = torch.argmax(outputs, dim = 1)
    accuracy = (preds == labels).float().mean().item() * 100

    total_accuracy += accuracy

    if idx % 2000 == 0:
      print("Iter [%d/%d]. Accuracy: %0.2f" % (idx + 1, len(test_loader), accuracy))

print("Final Accuracy: %0.2f" % (total_accuracy / len(test_loader)))

MyLinearNetwork(
  (linear_1): MyLinearLayer()
  (linear_2): MyLinearLayer()
  (softmax_fn): Softmax(dim=1)
)
Epoch [1/3]. Iter [1/1875]. Loss: 17.80. Accuracy: 9.38
Epoch [1/3]. Iter [101/1875]. Loss: 0.84. Accuracy: 81.25
Epoch [1/3]. Iter [201/1875]. Loss: 0.46. Accuracy: 87.50
Epoch [1/3]. Iter [301/1875]. Loss: 0.83. Accuracy: 62.50
Epoch [1/3]. Iter [401/1875]. Loss: 0.80. Accuracy: 78.12
Epoch [1/3]. Iter [501/1875]. Loss: 0.34. Accuracy: 87.50
Epoch [1/3]. Iter [601/1875]. Loss: 0.19. Accuracy: 90.62
Epoch [1/3]. Iter [701/1875]. Loss: 0.29. Accuracy: 87.50
Epoch [1/3]. Iter [801/1875]. Loss: 1.12. Accuracy: 87.50
Epoch [1/3]. Iter [901/1875]. Loss: 0.51. Accuracy: 84.38
Epoch [1/3]. Iter [1001/1875]. Loss: 0.26. Accuracy: 87.50
Epoch [1/3]. Iter [1101/1875]. Loss: 0.73. Accuracy: 81.25
Epoch [1/3]. Iter [1201/1875]. Loss: 0.38. Accuracy: 84.38
Epoch [1/3]. Iter [1301/1875]. Loss: 0.26. Accuracy: 90.62
Epoch [1/3]. Iter [1401/1875]. Loss: 0.34. Accuracy: 90.62
Epoch [1/3]. Iter